In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os
from pyspark.sql import functions as F
from tqdm.auto import tqdm
from fonctions.TMDB_data import download_info, download_person


In [0]:
df = spark.read.csv('/Volumes/films/bronze/landing/mapping/mapping_tmdb.csv',header=True,inferSchema=True)
df = df.filter(F.col("id") != -1)
for row in tqdm(df.collect()):
    name = row["Titre"]
    id = row["id"]
    type = row["Type"]

    path = f'/Volumes/films/bronze/landing/TMDB/{type}/{id}.json'

    if os.path.exists(path):
        tqdm.write("already downloaded")
    else:
        tqdm.write(f"downloading {name}")
        download_info(id, type)

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

In [0]:
df_film_crew = spark.read.json('/Volumes/films/bronze/landing/TMDB/Film_crew/')
df_serie_crew = spark.read.json('/Volumes/films/bronze/landing/TMDB/Série_crew/')

ids_film_crew = df_film_crew.select(F.explode("crew.id").alias("id"))
ids_film_cast = df_film_crew.select(F.explode("cast.id").alias("id"))
ids_serie_crew = df_serie_crew.select(F.explode("crew.id").alias("id"))
ids_serie_cast = df_serie_crew.select(F.explode("cast.id").alias("id"))

a_telecharger = ids_film_cast.union(ids_film_crew).union(ids_serie_cast).union(ids_serie_crew).distinct()

already_dl = {int(f.replace(".json","")) for f in os.listdir('/Volumes/films/bronze/landing/TMDB/person')}   ##on prefere un set car il est ordonné, et ca ira plus vite pour check si il est dans le set

ids = [person["id"] for person in a_telecharger.collect() if person["id"] not in already_dl]

with ThreadPoolExecutor(max_workers=20) as executor:
    futures = {}
    for i in range (len(ids)):
        futures[executor.submit(download_person, ids[i])]=ids[i]
    
    for future in tqdm(as_completed(futures), total = len(futures)):
        future.result()
